<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 1: One-hot과 Dense Vector 비교


In [ ]:
# 벡터와 수학 계산을 위한 NumPy 라이브러리
import numpy as np


# 두 벡터의 코사인 유사도를 계산하는 함수
def cosine(a, b):

    # 입력값을 NumPy 실수형 배열로 변환
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    # 코사인 유사도 계산
    # 값이 1에 가까울수록 두 벡터의 방향이 비슷함
    return float(
        np.dot(a, b) /
        (np.linalg.norm(a) * np.linalg.norm(b))
    )


# 사용할 단어 목록
vocab = ["고양이", "강아지", "자동차"]


# 원-핫 벡터 생성
# 각 단어는 서로 다른 위치만 1이고 나머지는 0
onehot = np.eye(len(vocab))

# 고양이와 강아지의 원-핫 벡터 유사도 계산
print(cosine(onehot[0], onehot[1]))


# 각 단어의 의미를 간단한 3차원 벡터로 표현한 예제
emb = {
    "고양이": [0.90, 0.80, 0.10],
    "강아지": [0.85, 0.90, 0.15],
    "자동차": [0.10, 0.05, 0.95]
}


# 고양이와 강아지의 임베딩 벡터 유사도 계산
print(cosine(emb["고양이"], emb["강아지"]))

# 실습 2: 임베딩 메모리 계산

In [ ]:
# V : 전체 단어(토큰)의 개수
# D : 임베딩 벡터의 차원
# bytes_per_float : 실수 1개(float32)를 저장하는 데 필요한 바이트
V = 200000
D = 1024
bytes_per_float = 4


# 원-핫 벡터를 V × V 행렬로 저장한다고 가정했을 때 필요한 메모리
# 200,000개의 단어 × 200,000차원
# 결과를 GiB 단위로 변환
onehot_identity_gib = (
    V * V * bytes_per_float / 1024**3
)


# 임베딩 테이블을 V × D 행렬로 저장할 때 필요한 메모리
# 200,000개의 단어 × 1,024차원
# 결과를 GiB 단위로 변환
embedding_gib = (
    V * D * bytes_per_float / 1024**3
)


# 원-핫 행렬의 메모리 크기 출력
print(f"Dense VxV identity: {onehot_identity_gib:.2f} GiB")

# 임베딩 테이블의 메모리 크기 출력
print(f"Embedding table: {embedding_gib:.2f} GiB")

# 임베딩 테이블에 들어 있는 전체 파라미터 수 출력
print(f"Embedding parameter 수: {V * D:,}")

# 실습 3: Word2Vec 직접 학습

In [ ]:
# gensim 라이브러리 설치
!pip install gensim

In [ ]:
# Word2Vec 모델 불러오기
from gensim.models import Word2Vec


# Word2Vec 학습에 사용할 간단한 문장들
# split()을 사용하여 문장을 단어 단위로 나눔
corpus = [
    "왕 은 나라 를 다스린다".split(),
    "여왕 은 나라 를 다스린다".split(),
    "고양이 는 귀여운 동물 이다".split(),
    "자동차 는 도로 를 달린다".split(),
    "사과 는 맛있는 과일 이다".split(),
    "포도 는 달콤한 과일 이다".split(),
    "강아지 는 사람 의 친구 이다".split(),
] * 100    # 작은 데이터이므로 같은 문장을 100번 반복


# Word2Vec 모델 학습
model = Word2Vec(
    corpus,

    vector_size=50,  # 각 단어를 50차원 벡터로 표현
    window=3,        # 주변 3개 단어까지 문맥으로 사용
    min_count=1,     # 1번 이상 등장한 모든 단어를 학습
    sg=1,            # Skip-gram 방식 사용
    negative=5,      # Negative Sampling에서 음성 샘플 5개 사용
    epochs=100       # 전체 학습 데이터를 100번 반복 학습
)


# 단어 쌍의 코사인 유사도 계산
for a, b in [
    ("사과", "포도"),
    ("고양이", "강아지"),
    ("고양이", "자동차")
]:

    # 두 단어의 임베딩 벡터 사이의 유사도 출력
    print(a, b, model.wv.similarity(a, b))

# 실습 4: CBOW vs Skip-gram 비교

In [ ]:
# CBOW와 Skip-gram에서 공통으로 사용할 학습 설정
common = dict(
    sentences=corpus,  # Word2Vec 학습에 사용할 문장 데이터
    vector_size=50,    # 단어를 50차원 벡터로 표현
    window=3,          # 주변 3개 단어까지 문맥으로 사용
    min_count=1,       # 1번 이상 등장한 모든 단어를 학습
    negative=5,        # Negative Sampling에서 음성 샘플 5개 사용
    epochs=100         # 전체 데이터를 100번 반복 학습
)


# CBOW 방식으로 Word2Vec 학습
# sg=0 : CBOW
# 주변 단어들을 보고 가운데 단어를 예측
cbow = Word2Vec(**common, sg=0)


# Skip-gram 방식으로 Word2Vec 학습
# sg=1 : Skip-gram
# 가운데 단어를 보고 주변 단어들을 예측
skip = Word2Vec(**common, sg=1)


# 두 모델에서 단어 쌍의 유사도를 각각 계산
for a, b in [
    ("사과", "포도"),
    ("고양이", "강아지")
]:

    print(
        a, b,

        # CBOW로 학습한 단어 벡터의 유사도
        "CBOW=", cbow.wv.similarity(a, b),

        # Skip-gram으로 학습한 단어 벡터의 유사도
        "Skip=", skip.wv.similarity(a, b)
    )

# 실습 5: FastText의 OOV (Out-of-Vocabulary)

In [ ]:
# Word2Vec과 FastText 모델 불러오기
from gensim.models import Word2Vec, FastText


# 학습에 사용할 간단한 문장 데이터
# 각 문장을 단어 단위로 나누어 리스트로 만듦
corpus_ft = [
    "자연어처리 는 인공지능 의 중요한 기술 이다".split(),
    "자연어처리 는 사람 의 언어 를 컴퓨터 로 처리한다".split(),
    "자연어처리 기술 을 이용하여 문서 를 분석한다".split(),
    "인공지능 은 다양한 분야 에 활용된다".split(),
    "컴퓨터 는 사람 의 언어 를 분석한다".split(),
] * 100    # 작은 데이터이므로 반복하여 학습


# -------------------------------------------------
# 1. Word2Vec 모델 학습
# -------------------------------------------------
w2v = Word2Vec(
    corpus_ft,
    vector_size=50,  # 단어를 50차원 벡터로 표현
    window=3,        # 주변 3개 단어까지 문맥으로 사용
    min_count=1,     # 1번 이상 등장한 모든 단어 학습
    sg=1,            # Skip-gram 방식 사용
    epochs=100       # 100번 반복 학습
)


# -------------------------------------------------
# 2. FastText 모델 학습
# -------------------------------------------------
ft = FastText(
    corpus_ft,
    vector_size=50,  # 단어를 50차원 벡터로 표현
    window=3,        # 주변 3개 단어까지 문맥으로 사용
    min_count=1,
    sg=1,            # Skip-gram 방식 사용
    epochs=100,

    # 단어를 2~4글자의 subword로 나누어 학습
    min_n=2,
    max_n=4
)


# 학습 데이터에 없는 새로운 단어
# "자연어처리"는 있지만 "자연어처리기"는 없음
oov = "자연어처리기"


# -------------------------------------------------
# 3. Word2Vec의 OOV 처리
# -------------------------------------------------
print("[Word2Vec]")

try:
    # 학습하지 않은 단어이므로 오류 발생
    print(w2v.wv[oov][:5])

except KeyError as e:
    print("OOV 단어라서 벡터가 없습니다.")
    print(e)


# -------------------------------------------------
# 4. FastText의 OOV 처리
# -------------------------------------------------
print("\n[FastText]")

# FastText는 subword를 이용하여
# 학습하지 않은 단어의 벡터도 만들 수 있음
print("벡터 앞 5개:")
print(ft.wv[oov][:5])


# -------------------------------------------------
# 5. OOV 단어와 비슷한 단어 찾기
# -------------------------------------------------
print("\n비슷한 단어:")

print(ft.wv.most_similar(oov, topn=5))

# 실습 6: BERT Contextual Represenation 관찰

In [ ]:
# Hugging Face에서 토크나이저와 BERT 모델 불러오기
from transformers import AutoTokenizer, AutoModel

# PyTorch 라이브러리
import torch


# 다국어 BERT(mBERT)의 토크나이저 불러오기
tok = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)

# 다국어 BERT 모델 불러오기
model = AutoModel.from_pretrained(
    "bert-base-multilingual-cased"
)


# '배'가 서로 다른 의미로 사용된 두 문장
sentences = [
    "나는 배를 타고 제주도에 갔다.",       # 배 = boat
    "점심을 많이 먹어서 배가 부르다."      # 배 = stomach
]


# 문장을 하나씩 BERT에 입력
for s in sentences:

    # 문장을 토큰화하고 PyTorch 텐서로 변환
    inputs = tok(
        s,
        return_tensors="pt"
    )

    # BERT에 문장을 입력하여 각 토큰의 임베딩을 얻음
    # last_hidden_state : 각 토큰의 문맥을 반영한 벡터
    hidden = model(**inputs).last_hidden_state[0]

    # BERT가 문장을 어떻게 토큰화했는지 출력
    print(tok.convert_ids_to_tokens(
        inputs["input_ids"][0]
    ))

    # 임베딩 텐서의 크기 출력
    print("hidden shape:", tuple(hidden.shape))

In [ ]:
# Hugging Face에서 토크나이저와 BERT 모델 불러오기
from transformers import AutoTokenizer, AutoModel

# 벡터 계산을 위한 PyTorch
import torch
import torch.nn.functional as F


# -------------------------------------------------
# 1. 다국어 BERT(mBERT) 불러오기
# -------------------------------------------------

# 문장을 토큰으로 나누는 토크나이저
tok = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)

# 문맥 임베딩을 생성하는 BERT 모델
model = AutoModel.from_pretrained(
    "bert-base-multilingual-cased"
)

# 추론 모드로 설정
# Dropout을 끄고 항상 같은 결과가 나오도록 함
model.eval()


# -------------------------------------------------
# 2. '배'의 의미가 다른 두 문장
# -------------------------------------------------

sentences = [
    "나는 배를 타고 제주도에 갔다.",      # 배 = boat
    "점심을 많이 먹어서 배가 부르다."     # 배 = stomach
]


# 각 문장에서 추출한 '배' 벡터를 저장할 리스트
vectors = []


# -------------------------------------------------
# 3. 각 문장에서 '배'의 문맥 임베딩 추출
# -------------------------------------------------

for s in sentences:

    # 문장을 토큰화하고 PyTorch 텐서로 변환
    inputs = tok(
        s,
        return_tensors="pt"
    )

    # 토큰 ID를 실제 토큰 문자열로 변환
    tokens = tok.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    # 학습이 아니라 추론만 하므로 gradient 계산하지 않음
    with torch.no_grad():

        # BERT를 통과시켜 각 토큰의 문맥 임베딩 생성
        hidden = model(**inputs).last_hidden_state[0]

    # '배' 토큰이 몇 번째 위치에 있는지 찾기
    idx = tokens.index("배")

    # 해당 위치의 768차원 벡터 추출
    vector = hidden[idx]

    # 나중에 비교하기 위해 저장
    vectors.append(vector)

    # 결과 확인
    print("\n문장:", s)
    print("토큰:", tokens)
    print("'배'의 위치:", idx)
    print("'배' 벡터 크기:", vector.shape)
    print("'배' 벡터 앞 5개:", vector[:5])


# -------------------------------------------------
# 4. 두 '배' 벡터의 코사인 유사도 계산
# -------------------------------------------------

similarity = F.cosine_similarity(
    vectors[0],
    vectors[1],
    dim=0
)

print("\n두 '배'의 코사인 유사도:")
print(f"{similarity.item():.4f}")

In [ ]:
## 중국어 실습
# Hugging Face에서 토크나이저와 BERT 모델 불러오기
from transformers import AutoTokenizer, AutoModel

# 벡터 계산을 위한 PyTorch
import torch
import torch.nn.functional as F


# -------------------------------------------------
# 1. 다국어 BERT(mBERT) 불러오기
# -------------------------------------------------

# 한국어, 중국어, 영어 등을 지원하는 mBERT 토크나이저
tok = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased"
)

# mBERT 모델 불러오기
model = AutoModel.from_pretrained(
    "bert-base-multilingual-cased"
)

# 추론 모드로 설정
model.eval()


# -------------------------------------------------
# 2. '苹果'가 서로 다른 의미로 사용된 중국어 문장
# -------------------------------------------------

sentences = [
    "我喜欢吃苹果。",        # 나는 사과를 먹는 것을 좋아한다.
    "苹果发布了新手机。"     # Apple이 새로운 스마트폰을 발표했다.
]


# 각 문장에서 '苹果'의 벡터를 저장할 리스트
vectors = []


# -------------------------------------------------
# 3. 각 문장에서 '苹果'의 문맥 임베딩 추출
# -------------------------------------------------

for s in sentences:

    # 문장을 토큰화하고 PyTorch 텐서로 변환
    inputs = tok(
        s,
        return_tensors="pt"
    )

    # 토큰 ID를 실제 토큰 문자열로 변환
    tokens = tok.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    # BERT를 이용하여 문맥 임베딩 계산
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]

    # 토큰화 결과 출력
    print("\n문장:", s)
    print("토큰:", tokens)

    # 중국어에서는 '苹果'가 여러 토큰으로 나뉠 수 있으므로
    # '苹'과 '果'의 위치를 찾음
    idx1 = tokens.index("苹")
    idx2 = tokens.index("果")

    # 두 글자의 벡터 평균을 '苹果'의 벡터로 사용
    vector = hidden[[idx1, idx2]].mean(dim=0)

    # 비교를 위해 저장
    vectors.append(vector)

    print("'苹果' 벡터 크기:", vector.shape)
    print("'苹果' 벡터 앞 5개:", vector[:5])


# -------------------------------------------------
# 4. 두 문장에서 '苹果' 벡터의 코사인 유사도 계산
# -------------------------------------------------

similarity = F.cosine_similarity(
    vectors[0],
    vectors[1],
    dim=0
)

print("\n두 '苹果' 문맥 임베딩의 코사인 유사도:")
print(f"{similarity.item():.4f}")

# 실습 7: Sentence Embedding으로 Semantic Search

In [ ]:
# 문장을 임베딩 벡터로 변환하기 위한 라이브러리
from sentence_transformers import SentenceTransformer

# 벡터 계산과 정렬을 위한 NumPy
import numpy as np


# -------------------------------------------------
# 1. 한국어 문장 임베딩 모델 불러오기
# -------------------------------------------------

model = SentenceTransformer("nlpai-lab/KURE-v1")


# -------------------------------------------------
# 2. 검색 대상이 되는 문서 준비
# -------------------------------------------------

docs = [
    "트랜스포머는 self-attention으로 토큰 관계를 계산한다.",
    "RAG는 관련 문서를 검색해 LLM 답변 근거로 제공한다.",
    "오늘 점심에는 김치찌개를 먹었다."
]


# -------------------------------------------------
# 3. 사용자의 질문
# -------------------------------------------------

query = "외부 자료를 찾아서 생성 모델에게 참고시키는 방법은?"


# -------------------------------------------------
# 4. 문서를 임베딩 벡터로 변환
# -------------------------------------------------

# 각 문장을 숫자로 이루어진 벡터로 변환
# normalize_embeddings=True:
# 벡터의 길이를 1로 정규화하여 유사도 계산을 쉽게 함
D = model.encode(
    docs,
    normalize_embeddings=True
)


# -------------------------------------------------
# 5. 질문도 임베딩 벡터로 변환
# -------------------------------------------------

q = model.encode(
    [query],
    normalize_embeddings=True
)[0]


# -------------------------------------------------
# 6. 질문과 각 문서의 유사도 계산
# -------------------------------------------------

# 정규화된 벡터의 내적(dot product)은
# 코사인 유사도와 같음
scores = D @ q


# -------------------------------------------------
# 7. 질문과 가장 비슷한 문서부터 출력
# -------------------------------------------------

# -scores : 높은 점수가 먼저 나오도록 내림차순 정렬
# [:3]    : 상위 3개 문서 선택
for i in np.argsort(-scores)[:3]:

    # 유사도 점수와 해당 문서 출력
    print(f"{scores[i]:.4f}  {docs[i]}")